# Cell 1 — Install fallback

In [1]:
from importlib.metadata import version, PackageNotFoundError
import subprocess, sys

REQS = {
    "segmentation-models-pytorch": "0.3.3",
    "timm": "0.9.2",
    "grad-cam": None,
    "scikit-image": None,
}

def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

current = {pkg: pkg_version(pkg) for pkg in REQS}
need_install = (
    current["segmentation-models-pytorch"] != "0.3.3"
    or current["timm"] != "0.9.2"
    or current["grad-cam"] is None
    or current["scikit-image"] is None
)

if need_install:
    print("Installing compatible package set...")
    print("Current versions:", current)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "segmentation-models-pytorch==0.3.3",
        "timm==0.9.2",
        "grad-cam",
        "scikit-image",
    ])
else:
    print("Compatible packages already available:", current)

import segmentation_models_pytorch as smp
from skimage.morphology import skeletonize
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, HiResCAM

print("Ready:", {
    "segmentation-models-pytorch": pkg_version("segmentation-models-pytorch"),
    "timm": pkg_version("timm"),
    "grad-cam": pkg_version("grad-cam"),
    "scikit-image": pkg_version("scikit-image"),
})

Installing compatible package set...
Current versions: {'segmentation-models-pytorch': None, 'timm': '1.0.25', 'grad-cam': None, 'scikit-image': '0.25.2'}
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 63.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 89.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cucim-cu12 26.2.0 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.


Ready: {'segmentation-models-pytorch': '0.3.3', 'timm': '0.9.2', 'grad-cam': '1.5.5', 'scikit-image': '0.26.0'}


# Cell 2 — Imports, config, output folders

In [2]:
import os
import gc
import json
import math
import random
import shutil
import warnings
from pathlib import Path
from contextlib import nullcontext

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, matthews_corrcoef

import segmentation_models_pytorch as smp
from tqdm.auto import tqdm
from skimage.morphology import skeletonize

warnings.filterwarnings("ignore")

SEED = 42
IMAGE_SIZE = 448
THRESHOLDS = np.round(np.arange(0.05, 1.00, 0.05), 2)
PIXEL_CM_SAMPLE_PER_IMAGE = 4096
MAX_CAM_EXAMPLES = 6

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/output/phase14a")
CACHE_DIR = OUT_DIR / "phase14a_prob_cache"
GT_CACHE_DIR = OUT_DIR / "phase14a_gt_cache"
PANEL_DIR = OUT_DIR / "phase14a_panels"
CAM_DIR = OUT_DIR / "phase14a_cam"
DEMO_DIR = OUT_DIR / "phase14a_demo_cam_update"

for p in [OUT_DIR, CACHE_DIR, GT_CACHE_DIR, PANEL_DIR, CAM_DIR, DEMO_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)

DEVICE: cuda
OUT_DIR: /kaggle/working/output/phase14a


# Cell 3 — Auto-discover Kaggle inputs

In [3]:
def find_file(filename: str) -> Path:
    matches = sorted(ROOT.rglob(filename))
    assert matches, f"Could not find file: {filename}"
    return matches[0]

def find_dir_named(dirname: str, required_children=None) -> Path:
    required_children = required_children or []
    matches = []
    for p in ROOT.rglob(dirname):
        if p.is_dir() and all((p / child).exists() for child in required_children):
            matches.append(p)
    assert matches, f"Could not find directory: {dirname}"
    return matches[0]

CRACK_ROOT = find_dir_named("crack_segmentation_dataset", required_children=["images", "masks"])
CRACK_IMAGES_DIR = CRACK_ROOT / "images"
CRACK_MASKS_DIR = CRACK_ROOT / "masks"

VAL_SPLIT = find_file("val_split.csv")
TRAIN_SPLIT = find_file("train_split.csv")
PHASE11B_FINALIST = find_file("phase11b_finalist_selection.json")
PHASE12_TEMP = find_file("phase12_temperature_fit.json")
PHASE12_OP = find_file("phase12_final_operating_point.json")
PHASE13_REPORT = find_file("phase13_detection_report.json")
PHASE13_SUMMARY = find_file("phase13_val_detection_summary.csv")
PHASE11_ROOT = find_dir_named("Phase-11")

# optional phase13 mask folder if present
phase13_mask_dirs = [p for p in ROOT.rglob("phase13_val_masks") if p.is_dir()]
PHASE13_MASK_DIR = phase13_mask_dirs[0] if len(phase13_mask_dirs) > 0 else None

print("CRACK_ROOT:", CRACK_ROOT)
print("VAL_SPLIT:", VAL_SPLIT)
print("PHASE11B_FINALIST:", PHASE11B_FINALIST)
print("PHASE12_TEMP:", PHASE12_TEMP)
print("PHASE12_OP:", PHASE12_OP)
print("PHASE13_REPORT:", PHASE13_REPORT)
print("PHASE13_SUMMARY:", PHASE13_SUMMARY)
print("PHASE11_ROOT:", PHASE11_ROOT)
print("PHASE13_MASK_DIR:", PHASE13_MASK_DIR)

CRACK_ROOT: /kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset
VAL_SPLIT: /kaggle/input/datasets/arman00p/phase-3-csv-files/val_split.csv
PHASE11B_FINALIST: /kaggle/input/datasets/arman00p/phase11b-finalist-selection-json/phase11b_finalist_selection.json
PHASE12_TEMP: /kaggle/input/datasets/arman00p/phase12-calibration-reliability-finalistt/Phase12_Calibration_Reliability_Finalist/phase12_temperature_fit.json
PHASE12_OP: /kaggle/input/datasets/arman00p/phase12-calibration-reliability-finalistt/Phase12_Calibration_Reliability_Finalist/phase12_final_operating_point.json
PHASE13_REPORT: /kaggle/input/datasets/arman00p/phase13-detection-outputs-public-demos/Phase13_Detection_Outputs_Public_Demo/phase13_detection_report.json
PHASE13_SUMMARY: /kaggle/input/datasets/arman00p/phase13-detection-outputs-public-demos/Phase13_Detection_Outputs_Public_Demo/phase13_val_detection_summary.csv
PHASE11_ROOT: /kaggle/input/datasets/arman00p/phase11-cracksafe-mrc-tr

# Cell 4 — Read control files and resolve validation dataframe

In [4]:
with open(PHASE11B_FINALIST, "r") as f:
    finalist_info = json.load(f)

with open(PHASE12_TEMP, "r") as f:
    phase12_temp = json.load(f)

with open(PHASE12_OP, "r") as f:
    phase12_op = json.load(f)

with open(PHASE13_REPORT, "r") as f:
    phase13_report = json.load(f)

val_df = pd.read_csv(VAL_SPLIT).copy()
phase13_summary_df = pd.read_csv(PHASE13_SUMMARY).copy()

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

img_col = pick_col(val_df, ["image_path", "img_path", "image", "filename", "file_name", "name", "id"])
mask_col = pick_col(val_df, ["mask_path", "mask", "mask_name"])
has_crack_col = pick_col(val_df, ["has_crack", "label", "target"])
source_col = pick_col(val_df, ["source", "dataset_source", "domain"])

assert has_crack_col is not None, "Could not find has_crack column"
assert img_col is not None or mask_col is not None, "Could not infer filename column"

name_source_col = img_col if img_col is not None else mask_col
val_df["filename"] = val_df[name_source_col].astype(str).map(lambda x: Path(x).name)
val_df["has_crack"] = val_df[has_crack_col].astype(int)
val_df["source"] = val_df[source_col].astype(str) if source_col is not None else "unknown"
val_df["image_path_resolved"] = val_df["filename"].map(lambda x: str(CRACK_IMAGES_DIR / x))
val_df["mask_path_resolved"] = val_df["filename"].map(lambda x: str(CRACK_MASKS_DIR / x))

assert val_df["image_path_resolved"].map(lambda p: Path(p).exists()).all(), "Missing validation images"
assert val_df["mask_path_resolved"].map(lambda p: Path(p).exists()).all(), "Missing validation masks"

MASK_THRESHOLD = float(phase12_op["recommended_mask_threshold"])
TEMPERATURE = float(phase12_temp["temperature"])
ENCODER_NAME = str(phase12_op["encoder_name"])

APPLY_PACKAGING_CLOSE = bool(phase13_report["apply_packaging_close"])
PACKAGING_CLOSE_KERNEL = int(phase13_report["packaging_close_kernel"])
PACKAGING_MIN_COMPONENT_AREA = int(phase13_report["packaging_min_component_area"])

print("Selected ablation:", finalist_info["selected_ablation"])
print("Mask threshold:", MASK_THRESHOLD)
print("Temperature:", TEMPERATURE)
print("Packaging close:", APPLY_PACKAGING_CLOSE)
print("Packaging min area:", PACKAGING_MIN_COMPONENT_AREA)

val_df.head()

Selected ablation: Boundary + clDice + Lovasz
Mask threshold: 0.55
Temperature: 1.5564953088760376
Packaging close: True
Packaging min area: 20


,stem,image_path,mask_path,source,empty_mask,has_crack,filename,image_path_resolved,mask_path_resolved
0,Volker_DSC01686_242_386_1564_1468,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...,volker,False,1,Volker_DSC01686_242_386_1564_1468.jpg,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...
1,GAPS384_train_1166_541_641,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...,gaps384,False,1,GAPS384_train_1166_541_641.jpg,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...
2,cracktree200_6243,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...,cracktree200,False,1,cracktree200_6243.jpg,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...
3,Rissbilder_for_Florian_9S6A3121_813_2194_2441_...,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...,rissbilder,False,1,Rissbilder_for_Florian_9S6A3121_813_2194_2441_...,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...
4,noncrack_noncrack_concrete_wall_23_9.jpg,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...,noncrack,True,0,noncrack_noncrack_concrete_wall_23_9.jpg.jpg,/kaggle/input/datasets/lakshaymiddha/crack-seg...,/kaggle/input/datasets/lakshaymiddha/crack-seg...


# Cell 5 — Resolve checkpoint and load the finalist model

In [5]:
def normalize_name(s: str) -> str:
    return "".join(ch.lower() for ch in str(s) if ch.isalnum())

checkpoint_from_phase12 = Path(phase12_op["checkpoint_path"])
if checkpoint_from_phase12.exists():
    BEST_CKPT = checkpoint_from_phase12
else:
    label_to_folder_key = {
        "Baseline": "ablationallfalse",
        "Boundary": "ablationonetrue",
        "Boundary + clDice": "ablationalltwotrue",
        "Boundary + clDice + Lovasz": "ablationallthreetrue",
        "Boundary + clDice + Lovasz + FDA + MRC": "ablationalltrue",
    }
    selected_ablation = finalist_info["selected_ablation"]
    folder_key = label_to_folder_key[selected_ablation]

    selected_run_dir = None
    for p in sorted(PHASE11_ROOT.iterdir()):
        if p.is_dir() and folder_key in normalize_name(p.name):
            selected_run_dir = p
            break

    assert selected_run_dir is not None, f"Could not resolve selected run dir for {selected_ablation}"
    best_ckpts = sorted(selected_run_dir.glob("*_best.pt"))
    assert best_ckpts, f"No *_best.pt found in {selected_run_dir}"
    BEST_CKPT = best_ckpts[0]

print("BEST_CKPT:", BEST_CKPT)

def extract_state_dict(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                state_dict = ckpt_obj[key]
                break
        else:
            if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
                state_dict = ckpt_obj
            else:
                raise ValueError("Could not locate state dict in checkpoint")
    else:
        raise ValueError("Unsupported checkpoint format")

    cleaned = {}
    for k, v in state_dict.items():
        nk = k
        for prefix in ["module.", "model."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v
    return cleaned

def build_unetpp_model(encoder_name):
    return smp.UnetPlusPlus(
        encoder_name=encoder_name,
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
    )

ckpt_obj = torch.load(str(BEST_CKPT), map_location="cpu")
state_dict = extract_state_dict(ckpt_obj)

encoder_candidates = [ENCODER_NAME, "efficientnet-b3", "timm-efficientnet-b3"]
encoder_candidates = list(dict.fromkeys(encoder_candidates))

best_model = None
best_encoder = None
best_preprocess_fn = None
best_score = None

for enc in encoder_candidates:
    try:
        model_try = build_unetpp_model(enc)
        incompat = model_try.load_state_dict(state_dict, strict=False)
        score = len(incompat.missing_keys) + len(incompat.unexpected_keys)
        print(f"Encoder={enc} | missing={len(incompat.missing_keys)} | unexpected={len(incompat.unexpected_keys)} | score={score}")
        if best_score is None or score < best_score:
            best_score = score
            best_model = model_try
            best_encoder = enc
            best_preprocess_fn = smp.encoders.get_preprocessing_fn(enc, "imagenet")
    except Exception as e:
        print(f"Encoder={enc} failed -> {e}")

assert best_model is not None, "Failed to load model"
model = best_model.to(DEVICE).eval()

print("\nChosen encoder:", best_encoder)
print("Load score:", best_score)

BEST_CKPT: /kaggle/input/datasets/arman00p/phase11-cracksafe-mrc-training-ablation/Phase-11/phase11-cracksafe-mrc-training_ablationAll_ThreeTrue/phase11_full_cracksafe_mrc_unetpp_effb3_448_best.pt
Encoder=efficientnet-b3 | missing=0 | unexpected=0 | score=0
Encoder=timm-efficientnet-b3 | missing=494 | unexpected=572 | score=1066

Chosen encoder: efficientnet-b3
Load score: 0


# Cell 6 — Helper functions# 

In [6]:
def read_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def read_mask_resized(path, image_size=448):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(path)
    mask = cv2.resize(mask, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.uint8)
    return mask

def preprocess_rgb_for_model(rgb_img, preprocess_fn, image_size=448):
    resized = cv2.resize(rgb_img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
    x = preprocess_fn(resized.astype(np.float32))
    x = np.transpose(x, (2, 0, 1)).astype(np.float32)
    return x

@torch.no_grad()
def predict_prob_map_448(rgb_img, temperature, preprocess_fn, image_size=448):
    x = preprocess_rgb_for_model(rgb_img, preprocess_fn, image_size=image_size)
    x = torch.from_numpy(x).unsqueeze(0).to(DEVICE)

    amp_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
    with amp_ctx:
        logits = model(x).float()[0, 0]

    prob = torch.sigmoid(logits / temperature).cpu().numpy().astype(np.float32)
    return prob

def clean_binary_mask(bin_mask, do_close=True, close_kernel=3, min_component_area=20):
    out = bin_mask.astype(np.uint8).copy()

    if do_close and close_kernel > 1:
        kernel = np.ones((close_kernel, close_kernel), np.uint8)
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, kernel)

    if min_component_area > 0:
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(out, connectivity=8)
        cleaned = np.zeros_like(out)
        for lab in range(1, num_labels):
            area = int(stats[lab, cv2.CC_STAT_AREA])
            if area >= min_component_area:
                cleaned[labels == lab] = 1
        out = cleaned

    return out

def empty_aware_iou_dice(pred_bool, gt_bool):
    pred_sum = pred_bool.sum()
    gt_sum = gt_bool.sum()

    if pred_sum == 0 and gt_sum == 0:
        return 1.0, 1.0
    if gt_sum == 0 and pred_sum > 0:
        return 0.0, 0.0

    inter = np.logical_and(pred_bool, gt_bool).sum()
    union = np.logical_or(pred_bool, gt_bool).sum()
    iou = inter / max(union, 1)
    dice = (2 * inter) / max(pred_sum + gt_sum, 1)
    return float(iou), float(dice)

def mask_to_boundary(mask, dilation_ratio=0.005):
    mask = mask.astype(np.uint8)
    if mask.sum() == 0:
        return np.zeros_like(mask)
    h, w = mask.shape
    diag = math.sqrt(h*h + w*w)
    d = max(1, int(round(dilation_ratio * diag)))
    kernel = np.ones((3, 3), np.uint8)

    padded = np.pad(mask, ((1, 1), (1, 1)), mode="constant", constant_values=0)
    eroded = cv2.erode(padded, kernel, iterations=d)[1:-1, 1:-1]
    boundary = (mask - eroded) > 0
    return boundary.astype(np.uint8)

def boundary_iou_score(pred_mask, gt_mask, dilation_ratio=0.005):
    pred_b = mask_to_boundary(pred_mask, dilation_ratio=dilation_ratio)
    gt_b = mask_to_boundary(gt_mask, dilation_ratio=dilation_ratio)
    return empty_aware_iou_dice(pred_b.astype(bool), gt_b.astype(bool))[0]

def cldice_score(pred_mask, gt_mask):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    if pred.sum() == 0 and gt.sum() == 0:
        return 1.0
    if pred.sum() == 0 or gt.sum() == 0:
        return 0.0

    skel_pred = skeletonize(pred)
    skel_gt = skeletonize(gt)

    tprec = (skel_pred & gt).sum() / max(skel_pred.sum(), 1)
    tsens = (skel_gt & pred).sum() / max(skel_gt.sum(), 1)

    if (tprec + tsens) == 0:
        return 0.0
    return float(2 * tprec * tsens / (tprec + tsens))

def plot_heatmap(values, row_labels, col_labels, title, save_path, fmt=".3f", cmap="viridis"):
    fig, ax = plt.subplots(figsize=(1.6 * len(col_labels) + 2, 0.55 * len(row_labels) + 2))
    im = ax.imshow(values, aspect="auto", cmap=cmap)
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.set_title(title)

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, format(values[i, j], fmt), ha="center", va="center", color="white", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.close()

print("Helpers ready")

Helpers ready


# Cell 7 — Cache probability maps and GT masks 

In [7]:
for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Caching prob maps"):
    filename = row["filename"]
    stem = Path(filename).stem

    prob_path = CACHE_DIR / f"{stem}.npy"
    gt_path = GT_CACHE_DIR / f"{stem}.npy"

    if prob_path.exists() and gt_path.exists():
        continue

    rgb = read_rgb(row["image_path_resolved"])
    prob = predict_prob_map_448(rgb, TEMPERATURE, best_preprocess_fn, IMAGE_SIZE)
    gt = read_mask_resized(row["mask_path_resolved"], IMAGE_SIZE)

    np.save(prob_path, prob.astype(np.float16))
    np.save(gt_path, gt.astype(np.uint8))

print("Cached probability maps:", len(list(CACHE_DIR.glob("*.npy"))))
print("Cached GT masks:", len(list(GT_CACHE_DIR.glob("*.npy"))))

Caching prob maps:   0%|          | 0/1695 [00:00<?, ?it/s]

Cached probability maps: 1695
Cached GT masks: 1695


# Cell 8 — Compute threshold metrics, per-image metrics, and confusion counts

In [8]:
threshold_acc = {
    float(t): {
        "count_all": 0,
        "sum_iou_all": 0.0,
        "sum_iou_crack": 0.0,
        "sum_dice_crack": 0.0,
        "count_crack": 0,
        "count_no_crack": 0,
        "fp_no_crack": 0,
        "img_tp": 0,
        "img_fp": 0,
        "img_tn": 0,
        "img_fn": 0,
    }
    for t in THRESHOLDS
}

selected_rows = []
pixel_cm_counts = np.zeros((2, 2), dtype=np.int64)  # rows=gt, cols=pred
rng = np.random.default_rng(SEED)

for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Metrics over cached probs"):
    filename = row["filename"]
    stem = Path(filename).stem
    has_crack = int(row["has_crack"])
    source = row["source"]

    prob = np.load(CACHE_DIR / f"{stem}.npy").astype(np.float32)
    gt = np.load(GT_CACHE_DIR / f"{stem}.npy").astype(np.uint8)

    for t in THRESHOLDS:
        pred = (prob >= float(t)).astype(np.uint8)
        if abs(float(t) - MASK_THRESHOLD) < 1e-9:
            pred = clean_binary_mask(
                pred,
                do_close=APPLY_PACKAGING_CLOSE,
                close_kernel=PACKAGING_CLOSE_KERNEL,
                min_component_area=PACKAGING_MIN_COMPONENT_AREA
            )

        iou, dice = empty_aware_iou_dice(pred.astype(bool), gt.astype(bool))
        pred_any = int(pred.sum() > 0)

        acc = threshold_acc[float(t)]
        acc["count_all"] += 1
        acc["sum_iou_all"] += iou

        if has_crack == 1:
            acc["count_crack"] += 1
            acc["sum_iou_crack"] += iou
            acc["sum_dice_crack"] += dice
        else:
            acc["count_no_crack"] += 1
            acc["fp_no_crack"] += pred_any

        if has_crack == 1 and pred_any == 1:
            acc["img_tp"] += 1
        elif has_crack == 0 and pred_any == 1:
            acc["img_fp"] += 1
        elif has_crack == 0 and pred_any == 0:
            acc["img_tn"] += 1
        elif has_crack == 1 and pred_any == 0:
            acc["img_fn"] += 1

        if abs(float(t) - MASK_THRESHOLD) < 1e-9:
            b_iou = boundary_iou_score(pred, gt)
            cld = cldice_score(pred, gt)

            selected_rows.append({
                "filename": filename,
                "source": source,
                "has_crack": has_crack,
                "pred_any": pred_any,
                "mean_iou": iou,
                "mean_dice": dice,
                "boundary_iou": b_iou,
                "cldice": cld,
                "crack_area_px": int(pred.sum()),
                "crack_area_ratio": float(pred.mean()),
                "max_prob": float(prob.max()),
                "mean_prob_pred": float(prob[pred > 0].mean()) if pred.any() else 0.0,
                "p95_prob_pred": float(np.percentile(prob[pred > 0], 95)) if pred.any() else 0.0,
            })

            flat_gt = gt.reshape(-1)
            flat_pred = pred.reshape(-1)
            n = min(PIXEL_CM_SAMPLE_PER_IMAGE, flat_gt.shape[0])
            idx = rng.choice(flat_gt.shape[0], size=n, replace=False)
            gt_s = flat_gt[idx]
            pred_s = flat_pred[idx]

            pixel_cm_counts[0, 0] += int(((gt_s == 0) & (pred_s == 0)).sum())
            pixel_cm_counts[0, 1] += int(((gt_s == 0) & (pred_s == 1)).sum())
            pixel_cm_counts[1, 0] += int(((gt_s == 1) & (pred_s == 0)).sum())
            pixel_cm_counts[1, 1] += int(((gt_s == 1) & (pred_s == 1)).sum())

threshold_rows = []
for t in THRESHOLDS:
    acc = threshold_acc[float(t)]

    mean_iou_all = acc["sum_iou_all"] / max(acc["count_all"], 1)
    mean_iou_crack_only = acc["sum_iou_crack"] / max(acc["count_crack"], 1)
    mean_dice_crack_only = acc["sum_dice_crack"] / max(acc["count_crack"], 1)
    no_crack_fp_rate = acc["fp_no_crack"] / max(acc["count_no_crack"], 1)

    img_tp, img_fp, img_tn, img_fn = acc["img_tp"], acc["img_fp"], acc["img_tn"], acc["img_fn"]
    image_precision = img_tp / max(img_tp + img_fp, 1)
    image_recall = img_tp / max(img_tp + img_fn, 1)
    image_specificity = img_tn / max(img_tn + img_fp, 1)
    image_f1 = 2 * image_precision * image_recall / max(image_precision + image_recall, 1e-8)

    threshold_rows.append({
        "threshold": float(t),
        "mean_iou_all": mean_iou_all,
        "mean_iou_crack_only": mean_iou_crack_only,
        "mean_dice_crack_only": mean_dice_crack_only,
        "no_crack_fp_rate": no_crack_fp_rate,
        "image_precision": image_precision,
        "image_recall": image_recall,
        "image_specificity": image_specificity,
        "image_f1": image_f1,
        "img_tp": img_tp,
        "img_fp": img_fp,
        "img_tn": img_tn,
        "img_fn": img_fn,
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values("threshold").reset_index(drop=True)
per_image_df = pd.DataFrame(selected_rows)

threshold_df.to_csv(OUT_DIR / "phase14a_threshold_metrics.csv", index=False)
per_image_df.to_csv(OUT_DIR / "phase14a_per_image_metrics.csv", index=False)

print("threshold_df:", threshold_df.shape)
print("per_image_df:", per_image_df.shape)
display(threshold_df.head())
display(per_image_df.head())

Metrics over cached probs:   0%|          | 0/1695 [00:00<?, ?it/s]

threshold_df: (19, 13)
per_image_df: (1695, 13)


,threshold,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,no_crack_fp_rate,image_precision,image_recall,image_specificity,image_f1,img_tp,img_fp,img_tn,img_fn
0,0.05,0.459607,0.448229,0.604280,0.463303,0.935995,1.0,0.536697,0.966939,1477,101,117,0
1,0.10,0.520510,0.489008,0.641906,0.266055,0.962215,1.0,0.733945,0.980744,1477,58,160,0
2,0.15,0.550575,0.512000,0.662345,0.188073,0.972991,1.0,0.811927,0.986311,1477,41,177,0
3,0.20,0.569947,0.528138,0.676410,0.146789,0.978794,1.0,0.853211,0.989283,1477,32,186,0
4,0.25,0.585937,0.541072,0.687495,0.110092,0.984011,1.0,0.889908,0.991941,1477,24,194,0


,filename,source,has_crack,pred_any,mean_iou,mean_dice,boundary_iou,cldice,crack_area_px,crack_area_ratio,max_prob,mean_prob_pred,p95_prob_pred
0,Volker_DSC01686_242_386_1564_1468.jpg,volker,1,1,0.707151,0.828458,0.369036,0.778601,8620,0.042949,0.995117,0.814637,0.985352
1,GAPS384_train_1166_541_641.jpg,gaps384,1,1,0.647177,0.785802,0.590473,0.936285,2544,0.012675,0.993164,0.870283,0.984375
2,cracktree200_6243.jpg,cracktree200,1,1,0.303350,0.465493,0.303350,0.483755,2258,0.011250,0.947754,0.752135,0.902832
3,Rissbilder_for_Florian_9S6A3121_813_2194_2441_...,rissbilder,1,1,0.574008,0.729358,0.416495,0.754676,8896,0.044324,0.977051,0.745453,0.941895
4,noncrack_noncrack_concrete_wall_23_9.jpg.jpg,noncrack,0,0,1.000000,1.000000,1.000000,1.000000,0,0.000000,0.146240,0.000000,0.000000


# Cell 9 — Confusion matrices and image-level report

In [9]:
# Image-level confusion matrix at selected threshold
y_true_img = per_image_df["has_crack"].to_numpy().astype(int)
y_pred_img = per_image_df["pred_any"].to_numpy().astype(int)

img_cm = confusion_matrix(y_true_img, y_pred_img, labels=[0, 1])
img_cm_norm = img_cm / np.maximum(img_cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(img_cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred no-crack", "Pred crack"])
ax.set_yticklabels(["GT no-crack", "GT crack"])
ax.set_title("Phase-14A Image-level Confusion Matrix (normalized)")

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{img_cm[i, j]}\n({img_cm_norm[i, j]:.3f})", ha="center", va="center", color="black", fontsize=11)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(OUT_DIR / "phase14a_image_level_confusion_matrix.png", dpi=220, bbox_inches="tight")
plt.close()

# Pixel-level normalized confusion matrix at selected threshold
pixel_cm_norm = pixel_cm_counts / np.maximum(pixel_cm_counts.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pixel_cm_norm, cmap="Purples", vmin=0, vmax=1)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred background", "Pred crack"])
ax.set_yticklabels(["GT background", "GT crack"])
ax.set_title("Phase-14A Pixel-level Confusion Matrix (normalized)")

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{pixel_cm_counts[i, j]}\n({pixel_cm_norm[i, j]:.3f})", ha="center", va="center", color="black", fontsize=11)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(OUT_DIR / "phase14a_pixel_confusion_matrix_normalized.png", dpi=220, bbox_inches="tight")
plt.close()

image_report = {
    "threshold": MASK_THRESHOLD,
    "image_precision": float(precision_score(y_true_img, y_pred_img, zero_division=0)),
    "image_recall": float(recall_score(y_true_img, y_pred_img, zero_division=0)),
    "image_f1": float(f1_score(y_true_img, y_pred_img, zero_division=0)),
    "image_mcc": float(matthews_corrcoef(y_true_img, y_pred_img)),
    "num_images": int(len(y_true_img)),
    "tp": int(img_cm[1, 1]),
    "fp": int(img_cm[0, 1]),
    "tn": int(img_cm[0, 0]),
    "fn": int(img_cm[1, 0]),
}

with open(OUT_DIR / "phase14a_image_level_report.json", "w") as f:
    json.dump(image_report, f, indent=2)

print(json.dumps(image_report, indent=2))

{
  "threshold": 0.55,
  "image_precision": 0.9966261808367072,
  "image_recall": 1.0,
  "image_f1": 0.9983102399459277,
  "image_mcc": 0.9867967278997551,
  "num_images": 1695,
  "tp": 1477,
  "fp": 5,
  "tn": 213,
  "fn": 0
}


# Cell 10 — Source heatmap, threshold heatmap, boundary/topology summaries

In [10]:
# Source-wise summary
source_rows = []
for source, sdf in per_image_df.groupby("source"):
    y_true = sdf["has_crack"].to_numpy().astype(int)
    y_pred = sdf["pred_any"].to_numpy().astype(int)

    source_rows.append({
        "source": source,
        "mean_iou_crack_only": float(sdf.loc[sdf["has_crack"] == 1, "mean_iou"].mean()) if (sdf["has_crack"] == 1).any() else np.nan,
        "mean_dice_crack_only": float(sdf.loc[sdf["has_crack"] == 1, "mean_dice"].mean()) if (sdf["has_crack"] == 1).any() else np.nan,
        "no_crack_fp_rate": float(sdf.loc[sdf["has_crack"] == 0, "pred_any"].mean()) if (sdf["has_crack"] == 0).any() else np.nan,
        "boundary_iou": float(sdf.loc[sdf["has_crack"] == 1, "boundary_iou"].mean()) if (sdf["has_crack"] == 1).any() else np.nan,
        "cldice": float(sdf.loc[sdf["has_crack"] == 1, "cldice"].mean()) if (sdf["has_crack"] == 1).any() else np.nan,
        "image_precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "image_recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "count": int(len(sdf)),
    })

source_df = pd.DataFrame(source_rows).sort_values("source").reset_index(drop=True)
source_df.to_csv(OUT_DIR / "phase14a_source_metric_summary.csv", index=False)

# Boundary/topology summary
boundary_topology_summary = pd.DataFrame([
    {
        "scope": "overall_crack_only",
        "boundary_iou": float(per_image_df.loc[per_image_df["has_crack"] == 1, "boundary_iou"].mean()),
        "cldice": float(per_image_df.loc[per_image_df["has_crack"] == 1, "cldice"].mean()),
    }
] + [
    {
        "scope": f"source::{row['source']}",
        "boundary_iou": row["boundary_iou"],
        "cldice": row["cldice"],
    }
    for _, row in source_df.iterrows()
])

boundary_topology_summary.to_csv(OUT_DIR / "phase14a_boundary_topology_summary.csv", index=False)

# Source heatmap
heatmap_cols = [
    "mean_iou_crack_only",
    "mean_dice_crack_only",
    "no_crack_fp_rate",
    "boundary_iou",
    "cldice",
    "image_precision",
    "image_recall",
]
heatmap_vals = source_df[heatmap_cols].fillna(0.0).to_numpy()
plot_heatmap(
    heatmap_vals,
    row_labels=source_df["source"].tolist(),
    col_labels=heatmap_cols,
    title="Phase-14A Source-wise Metric Heatmap",
    save_path=OUT_DIR / "phase14a_source_metric_heatmap.png",
    fmt=".3f",
    cmap="viridis",
)

# Threshold heatmap
threshold_heatmap_cols = [
    "mean_iou_crack_only",
    "mean_dice_crack_only",
    "no_crack_fp_rate",
    "image_precision",
    "image_recall",
    "image_specificity",
    "image_f1",
]
th_vals = threshold_df[threshold_heatmap_cols].to_numpy()
plot_heatmap(
    th_vals,
    row_labels=[f"{t:.2f}" for t in threshold_df["threshold"]],
    col_labels=threshold_heatmap_cols,
    title="Phase-14A Threshold Metric Heatmap",
    save_path=OUT_DIR / "phase14a_threshold_metric_heatmap.png",
    fmt=".3f",
    cmap="magma",
)

# Boundary IoU barplot
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(source_df["source"], source_df["boundary_iou"])
ax.set_title("Phase-14A Boundary IoU by Source")
ax.set_ylabel("Boundary IoU")
ax.set_xticklabels(source_df["source"], rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "phase14a_boundary_iou_barplot.png", dpi=220, bbox_inches="tight")
plt.close()

# clDice barplot
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(source_df["source"], source_df["cldice"])
ax.set_title("Phase-14A clDice by Source")
ax.set_ylabel("clDice")
ax.set_xticklabels(source_df["source"], rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "phase14a_cldice_barplot.png", dpi=220, bbox_inches="tight")
plt.close()

print("Saved:")
print("-", OUT_DIR / "phase14a_source_metric_summary.csv")
print("-", OUT_DIR / "phase14a_boundary_topology_summary.csv")
print("-", OUT_DIR / "phase14a_source_metric_heatmap.png")
print("-", OUT_DIR / "phase14a_threshold_metric_heatmap.png")

Saved:
- /kaggle/working/output/phase14a/phase14a_source_metric_summary.csv
- /kaggle/working/output/phase14a/phase14a_boundary_topology_summary.csv
- /kaggle/working/output/phase14a/phase14a_source_metric_heatmap.png
- /kaggle/working/output/phase14a/phase14a_threshold_metric_heatmap.png


# Cell 11 — Main qualitative panel and failure panel

In [11]:
# Rebuild predicted mask for selected threshold using the cached probability map
def get_pred_mask_for_filename(filename):
    stem = Path(filename).stem
    prob = np.load(CACHE_DIR / f"{stem}.npy").astype(np.float32)
    pred = (prob >= MASK_THRESHOLD).astype(np.uint8)
    pred = clean_binary_mask(
        pred,
        do_close=APPLY_PACKAGING_CLOSE,
        close_kernel=PACKAGING_CLOSE_KERNEL,
        min_component_area=PACKAGING_MIN_COMPONENT_AREA
    )
    return prob, pred

# Select cases
best_tp = per_image_df[(per_image_df["has_crack"] == 1) & (per_image_df["pred_any"] == 1)].sort_values(
    ["mean_iou", "p95_prob_pred"], ascending=[False, False]
).head(2)

hard_tp = per_image_df[(per_image_df["has_crack"] == 1) & (per_image_df["pred_any"] == 1)].sort_values(
    ["mean_iou", "cldice"], ascending=[True, True]
).head(2)

fp_cases = per_image_df[(per_image_df["has_crack"] == 0) & (per_image_df["pred_any"] == 1)].sort_values(
    ["p95_prob_pred", "crack_area_ratio"], ascending=[False, False]
).head(1)

fn_cases = per_image_df[(per_image_df["has_crack"] == 1) & (per_image_df["pred_any"] == 0)].head(1)

main_examples_df = pd.concat([best_tp, hard_tp, fp_cases, fn_cases], axis=0).drop_duplicates("filename").head(6).reset_index(drop=True)

# Failure panel: more FP/FN/hard cases
failure_examples_df = pd.concat([
    fp_cases,
    per_image_df[(per_image_df["has_crack"] == 0) & (per_image_df["pred_any"] == 1)].sort_values(
        ["crack_area_ratio", "p95_prob_pred"], ascending=[False, False]
    ).head(2),
    fn_cases,
    per_image_df[(per_image_df["has_crack"] == 1) & (per_image_df["pred_any"] == 1)].sort_values(
        ["boundary_iou", "cldice"], ascending=[True, True]
    ).head(2),
], axis=0).drop_duplicates("filename").head(6).reset_index(drop=True)

main_examples_df.to_csv(OUT_DIR / "phase14a_main_panel_examples.csv", index=False)
failure_examples_df.to_csv(OUT_DIR / "phase14a_failure_panel_examples.csv", index=False)

def make_case_panel(case_df, save_path, title):
    n = len(case_df)
    cols = 4
    fig, axes = plt.subplots(n, cols, figsize=(4 * cols, 3.5 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, (_, row) in enumerate(case_df.iterrows()):
        filename = row["filename"]
        row_full = val_df[val_df["filename"] == filename].iloc[0]

        rgb = read_rgb(row_full["image_path_resolved"])
        gt = read_mask_resized(row_full["mask_path_resolved"], IMAGE_SIZE)
        _, pred = get_pred_mask_for_filename(filename)

        overlay = rgb.copy()
        green = np.zeros_like(rgb)
        green[..., 1] = 255
        overlay = np.where(pred[..., None] > 0, (0.65 * overlay + 0.35 * green), overlay).astype(np.uint8)

        title_text = f"{filename}\ngt={int(row['has_crack'])} pred={int(row['pred_any'])} IoU={row['mean_iou']:.3f}"

        axes[r, 0].imshow(rgb)
        axes[r, 0].set_title(title_text, fontsize=9)
        axes[r, 1].imshow(gt, cmap="gray")
        axes[r, 1].set_title("GT mask", fontsize=9)
        axes[r, 2].imshow(pred, cmap="gray")
        axes[r, 2].set_title("Pred mask", fontsize=9)
        axes[r, 3].imshow(overlay)
        axes[r, 3].set_title("Overlay", fontsize=9)

        for c in range(cols):
            axes[r, c].axis("off")

    fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.close()

make_case_panel(main_examples_df, OUT_DIR / "phase14a_qualitative_panel_main_paper.png", "Phase-14A Main Qualitative Panel")
make_case_panel(failure_examples_df, OUT_DIR / "phase14a_failure_panel_supplement.png", "Phase-14A Failure / Hard-case Panel")

print("Saved qualitative panels")

Saved qualitative panels


# Cell 12 — CAM imports, target layer, and CAM helpers

In [12]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, HiResCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

def find_target_layer(model):
    # preferred for efficientnet-b3 SMP encoder
    if hasattr(model.encoder, "_blocks") and len(model.encoder._blocks) > 0:
        return model.encoder._blocks[-1]

    # fallback: last Conv2d found
    last_conv = None
    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            last_conv = m
    assert last_conv is not None, "Could not find target conv layer for CAM"
    return last_conv

TARGET_LAYER = find_target_layer(model)
print("TARGET_LAYER:", TARGET_LAYER)

class BinarySegmentationTarget:
    def __init__(self, target_mask_448):
        self.target_mask = torch.from_numpy(target_mask_448.astype(np.float32))

    def __call__(self, model_output):
        if model_output.ndim == 4:
            out = model_output[0, 0]
        elif model_output.ndim == 3:
            out = model_output[0]
        else:
            raise ValueError(f"Unexpected model_output shape: {model_output.shape}")

        return (out * self.target_mask.to(out.device)).sum()

def get_cam_method(name, model, target_layer):
    if name == "HiResCAM":
        return HiResCAM(model=model, target_layers=[target_layer])
    elif name == "GradCAMPlusPlus":
        return GradCAMPlusPlus(model=model, target_layers=[target_layer])
    elif name == "GradCAM":
        return GradCAM(model=model, target_layers=[target_layer])
    else:
        raise ValueError(f"Unknown CAM method: {name}")

@torch.no_grad()
def get_prob_and_pred_from_filename(filename):
    stem = Path(filename).stem
    prob = np.load(CACHE_DIR / f"{stem}.npy").astype(np.float32)
    pred = (prob >= MASK_THRESHOLD).astype(np.uint8)
    pred = clean_binary_mask(
        pred,
        do_close=APPLY_PACKAGING_CLOSE,
        close_kernel=PACKAGING_CLOSE_KERNEL,
        min_component_area=PACKAGING_MIN_COMPONENT_AREA
    )
    return prob, pred

def make_cam_for_filename(filename, cam_method_name="HiResCAM"):
    row_full = val_df[val_df["filename"] == filename].iloc[0]
    rgb_orig = read_rgb(row_full["image_path_resolved"])
    rgb_448 = cv2.resize(rgb_orig, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)

    x = preprocess_rgb_for_model(rgb_orig, best_preprocess_fn, IMAGE_SIZE)
    input_tensor = torch.from_numpy(x).unsqueeze(0).to(DEVICE)

    prob_448, pred_448 = get_prob_and_pred_from_filename(filename)
    gt_448 = read_mask_resized(row_full["mask_path_resolved"], IMAGE_SIZE)

    if pred_448.sum() > 0:
        target_mask = pred_448
    elif gt_448.sum() > 0:
        target_mask = gt_448
    else:
        target_mask = (prob_448 >= np.percentile(prob_448, 99.5)).astype(np.uint8)

    cam_target = BinarySegmentationTarget(target_mask)

    with get_cam_method(cam_method_name, model, TARGET_LAYER) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=[cam_target])[0]

    rgb_float = rgb_448.astype(np.float32) / 255.0
    cam_vis = show_cam_on_image(rgb_float, grayscale_cam, use_rgb=True)

    return {
        "rgb_448": rgb_448,
        "gt_448": gt_448,
        "pred_448": pred_448,
        "grayscale_cam": grayscale_cam,
        "cam_vis": cam_vis,
    }

print("CAM helpers ready")

TARGET_LAYER: MBConvBlock(
  (_expand_conv): Conv2dStaticSamePadding(
    384, 2304, kernel_size=(1, 1), stride=(1, 1), bias=False
    (static_padding): Identity()
  )
  (_bn0): BatchNorm2d(2304, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_depthwise_conv): Conv2dStaticSamePadding(
    2304, 2304, kernel_size=(3, 3), stride=(1, 1), groups=2304, bias=False
    (static_padding): ZeroPad2d((1, 1, 1, 1))
  )
  (_bn1): BatchNorm2d(2304, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_se_reduce): Conv2dStaticSamePadding(
    2304, 96, kernel_size=(1, 1), stride=(1, 1)
    (static_padding): Identity()
  )
  (_se_expand): Conv2dStaticSamePadding(
    96, 2304, kernel_size=(1, 1), stride=(1, 1)
    (static_padding): Identity()
  )
  (_project_conv): Conv2dStaticSamePadding(
    2304, 384, kernel_size=(1, 1), stride=(1, 1), bias=False
    (static_padding): Identity()
  )
  (_bn2): BatchNorm2d(384, eps=0.001, momentum=0.

# Cell 13 — CAM galleries and comparison panel

In [13]:
cam_examples = main_examples_df["filename"].tolist()[:MAX_CAM_EXAMPLES]
if len(cam_examples) == 0:
    cam_examples = per_image_df["filename"].head(MAX_CAM_EXAMPLES).tolist()

def save_cam_gallery(method_name, filenames, save_path):
    n = len(filenames)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, filename in enumerate(filenames):
        out = make_cam_for_filename(filename, cam_method_name=method_name)

        axes[r, 0].imshow(out["rgb_448"])
        axes[r, 0].set_title(f"{filename}\nOriginal", fontsize=9)

        axes[r, 1].imshow(out["pred_448"], cmap="gray")
        axes[r, 1].set_title("Pred mask", fontsize=9)

        axes[r, 2].imshow(out["cam_vis"])
        axes[r, 2].set_title(method_name, fontsize=9)

        for c in range(3):
            axes[r, c].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.close()

save_cam_gallery("HiResCAM", cam_examples, OUT_DIR / "phase14a_hirescam_gallery.png")
save_cam_gallery("GradCAMPlusPlus", cam_examples, OUT_DIR / "phase14a_gradcampp_gallery.png")
save_cam_gallery("GradCAM", cam_examples, OUT_DIR / "phase14a_gradcam_gallery.png")

# Comparison panel: same cases, 5 columns
n = len(cam_examples)
fig, axes = plt.subplots(n, 5, figsize=(18, 4 * n))
if n == 1:
    axes = np.expand_dims(axes, axis=0)

for r, filename in enumerate(cam_examples):
    hires = make_cam_for_filename(filename, "HiResCAM")
    gpp = make_cam_for_filename(filename, "GradCAMPlusPlus")
    gcam = make_cam_for_filename(filename, "GradCAM")

    axes[r, 0].imshow(hires["rgb_448"])
    axes[r, 0].set_title(f"{filename}\nOriginal", fontsize=9)

    axes[r, 1].imshow(hires["gt_448"], cmap="gray")
    axes[r, 1].set_title("GT", fontsize=9)

    axes[r, 2].imshow(hires["pred_448"], cmap="gray")
    axes[r, 2].set_title("Pred", fontsize=9)

    axes[r, 3].imshow(hires["cam_vis"])
    axes[r, 3].set_title("HiResCAM", fontsize=9)

    axes[r, 4].imshow(gpp["cam_vis"])
    axes[r, 4].set_title("GradCAM++", fontsize=9)

    for c in range(5):
        axes[r, c].axis("off")

plt.tight_layout()
plt.savefig(OUT_DIR / "phase14a_cam_comparison_panel.png", dpi=220, bbox_inches="tight")
plt.close()

print("Saved CAM galleries")

Saved CAM galleries


# Cell 14 — Build updated demo package with CAM output

In [14]:
# choose examples for demo package
demo_examples = cam_examples[:4]

for filename in demo_examples:
    src = val_df.loc[val_df["filename"] == filename, "image_path_resolved"].iloc[0]
    dst_dir = DEMO_DIR / "examples"
    dst_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst_dir / Path(src).name)

demo_config = {
    "selected_ablation": finalist_info["selected_ablation"],
    "encoder_name": best_encoder,
    "checkpoint_path": "checkpoint.pt",
    "temperature": TEMPERATURE,
    "mask_threshold": MASK_THRESHOLD,
    "image_size": IMAGE_SIZE,
    "apply_packaging_close": APPLY_PACKAGING_CLOSE,
    "packaging_close_kernel": PACKAGING_CLOSE_KERNEL,
    "packaging_min_component_area": PACKAGING_MIN_COMPONENT_AREA,
}

with open(DEMO_DIR / "phase14a_demo_config.json", "w") as f:
    json.dump(demo_config, f, indent=2)

shutil.copy2(BEST_CKPT, DEMO_DIR / "checkpoint.pt")

examples_json = [str((Path("examples") / Path(f).name).as_posix()) for f in demo_examples]
with open(DEMO_DIR / "examples.json", "w") as f:
    json.dump(examples_json, f, indent=2)

app_py = r'''
import json
from pathlib import Path
from contextlib import nullcontext

import cv2
import numpy as np
import torch
import gradio as gr
import segmentation_models_pytorch as smp
from skimage.morphology import skeletonize
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, HiResCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

ROOT = Path(__file__).resolve().parent
with open(ROOT / "phase14a_demo_config.json", "r") as f:
    CFG = json.load(f)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def build_model():
    model = smp.UnetPlusPlus(
        encoder_name=CFG["encoder_name"],
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
    )
    ckpt = torch.load(ROOT / CFG["checkpoint_path"], map_location="cpu")

    if isinstance(ckpt, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                state_dict = ckpt[key]
                break
        else:
            state_dict = ckpt
    else:
        state_dict = ckpt

    cleaned = {}
    for k, v in state_dict.items():
        nk = k
        for prefix in ["module.", "model."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v

    model.load_state_dict(cleaned, strict=False)
    model = model.to(DEVICE).eval()
    preprocess_fn = smp.encoders.get_preprocessing_fn(CFG["encoder_name"], "imagenet")
    return model, preprocess_fn

MODEL, PREPROCESS_FN = build_model()

def preprocess_rgb(rgb_img):
    resized = cv2.resize(rgb_img, (CFG["image_size"], CFG["image_size"]), interpolation=cv2.INTER_LINEAR)
    x = PREPROCESS_FN(resized.astype(np.float32))
    x = np.transpose(x, (2, 0, 1)).astype(np.float32)
    return resized, x

def clean_binary_mask(bin_mask):
    out = bin_mask.astype(np.uint8).copy()

    if CFG["apply_packaging_close"] and CFG["packaging_close_kernel"] > 1:
        kernel = np.ones((CFG["packaging_close_kernel"], CFG["packaging_close_kernel"]), np.uint8)
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, kernel)

    min_area = int(CFG["packaging_min_component_area"])
    if min_area > 0:
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(out, connectivity=8)
        cleaned = np.zeros_like(out)
        for lab in range(1, num_labels):
            area = int(stats[lab, cv2.CC_STAT_AREA])
            if area >= min_area:
                cleaned[labels == lab] = 1
        out = cleaned

    return out

def find_target_layer(model):
    if hasattr(model.encoder, "_blocks") and len(model.encoder._blocks) > 0:
        return model.encoder._blocks[-1]
    last_conv = None
    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            last_conv = m
    return last_conv

TARGET_LAYER = find_target_layer(MODEL)

class BinarySegmentationTarget:
    def __init__(self, target_mask):
        self.target_mask = torch.from_numpy(target_mask.astype(np.float32))

    def __call__(self, model_output):
        if model_output.ndim == 4:
            out = model_output[0, 0]
        else:
            out = model_output[0]
        return (out * self.target_mask.to(out.device)).sum()

def get_cam_method(name):
    if name == "HiResCAM":
        return HiResCAM(model=MODEL, target_layers=[TARGET_LAYER])
    elif name == "GradCAMPlusPlus":
        return GradCAMPlusPlus(model=MODEL, target_layers=[TARGET_LAYER])
    else:
        return GradCAM(model=MODEL, target_layers=[TARGET_LAYER])

@torch.no_grad()
def predict_prob_map(input_tensor):
    amp_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
    with amp_ctx:
        logits = MODEL(input_tensor).float()[0, 0]
    prob = torch.sigmoid(logits / float(CFG["temperature"])).cpu().numpy().astype(np.float32)
    return prob

def build_components(prob_map, bin_mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask.astype(np.uint8), connectivity=8)
    components = []
    for lab in range(1, num_labels):
        comp_mask = (labels == lab).astype(np.uint8)
        ys, xs = np.where(comp_mask > 0)
        if len(xs) == 0:
            continue
        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        scores = prob_map[comp_mask > 0]
        score_p95 = float(np.percentile(scores, 95)) if scores.size > 0 else 0.0

        pts = np.column_stack([xs, ys]).astype(np.float32)
        rect = cv2.minAreaRect(pts)
        box = cv2.boxPoints(rect).astype(np.int32).tolist()

        components.append({
            "area_px": int(comp_mask.sum()),
            "score_p95": score_p95,
            "aabb": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
            "obb": {"points": box}
        })
    return components

def draw_overlay(rgb_img, bin_mask, components, alpha=0.35):
    overlay = rgb_img.copy()
    green = np.zeros_like(rgb_img)
    green[..., 1] = 255
    overlay = np.where(bin_mask[..., None] > 0, (1 - alpha) * overlay + alpha * green, overlay).astype(np.uint8)

    for i, comp in enumerate(components, start=1):
        a = comp["aabb"]
        cv2.rectangle(overlay, (a["x1"], a["y1"]), (a["x2"], a["y2"]), (255, 0, 0), 2)
        obb_pts = np.array(comp["obb"]["points"], dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(overlay, [obb_pts], True, (255, 255, 0), 2)
        cv2.putText(overlay, f"{i}:{comp['score_p95']:.2f}", (a["x1"], max(15, a["y1"] - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return overlay

def predict(image, cam_method, confidence_gate):
    if image is None:
        return None, None, None, {}, "Please upload an image."

    if image.ndim == 2:
        image = np.stack([image, image, image], axis=-1)

    rgb = image.astype(np.uint8)
    rgb_448, x = preprocess_rgb(rgb)
    input_tensor = torch.from_numpy(x).unsqueeze(0).to(DEVICE)

    prob = predict_prob_map(input_tensor)
    pred = (prob >= float(CFG["mask_threshold"])).astype(np.uint8)
    pred = clean_binary_mask(pred)

    if pred.sum() > 0:
        target_mask = pred
    else:
        target_mask = (prob >= np.percentile(prob, 99.5)).astype(np.uint8)

    target = BinarySegmentationTarget(target_mask)
    with get_cam_method(cam_method) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=[target])[0]

    cam_vis = show_cam_on_image(rgb_448.astype(np.float32) / 255.0, grayscale_cam, use_rgb=True)

    components = build_components(prob, pred)
    overlay = draw_overlay(rgb_448, pred, components)
    mask_rgb = np.stack([pred * 255, pred * 255, pred * 255], axis=-1).astype(np.uint8)

    pred_any = int(pred.sum() > 0)
    max_prob = float(prob.max()) if prob.size > 0 else 0.0
    crack_area_px = int(pred.sum())
    crack_area_ratio = float(pred.mean())
    gate_pass = int(max_prob >= confidence_gate)

    summary = {
        "pred_any": pred_any,
        "confidence_gate_pass": gate_pass,
        "max_calibrated_confidence": max_prob,
        "crack_area_px": crack_area_px,
        "crack_area_ratio": crack_area_ratio,
        "num_components": len(components),
        "cam_method": cam_method,
        "components": components
    }

    summary_md = f"""
### CrackSAFE-MRC Demo with Explainability
- **Predicted crack/no-crack:** {"Crack" if pred_any else "No-crack"}
- **Mask threshold:** {CFG["mask_threshold"]:.2f}
- **Confidence gate:** {confidence_gate:.2f}
- **Confidence gate passed:** {"Yes" if gate_pass else "No"}
- **Max calibrated confidence:** {max_prob:.4f}
- **Crack area (pixels):** {crack_area_px}
- **Crack area ratio:** {crack_area_ratio:.6f}
- **CAM method:** {cam_method}
- **Detected components:** {len(components)}
"""

    return overlay, mask_rgb, cam_vis, summary, summary_md

examples = []
examples_dir = ROOT / "examples"
if examples_dir.exists():
    for p in sorted(examples_dir.glob("*.jpg")):
        examples.append([str(p), "HiResCAM", 0.90])

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="numpy", label="Upload wall/concrete image"),
        gr.Dropdown(choices=["HiResCAM", "GradCAMPlusPlus", "GradCAM"], value="HiResCAM", label="CAM method"),
        gr.Slider(minimum=0.50, maximum=0.99, value=0.90, step=0.01, label="Confidence gate")
    ],
    outputs=[
        gr.Image(type="numpy", label="Overlay"),
        gr.Image(type="numpy", label="Binary mask"),
        gr.Image(type="numpy", label="Explanation heatmap"),
        gr.JSON(label="Detections"),
        gr.Markdown(label="Summary")
    ],
    title="CrackSAFE-MRC Demo + Explainability",
    description="Upload a wall or concrete image to get calibrated crack segmentation, detections, and a CAM explanation overlay.",
    examples=examples if len(examples) > 0 else None
)

if __name__ == "__main__":
    demo.launch()
'''

requirements_txt = """torch
torchvision
opencv-python-headless
numpy
pandas
segmentation-models-pytorch==0.3.3
timm==0.9.7
grad-cam
scikit-image
gradio==4.44.1
"""

readme_md = f"""# CrackSAFE-MRC Demo with Explainability

This package was generated from the Phase-14A Kaggle notebook.

## Included
- app.py
- checkpoint.pt
- phase14a_demo_config.json
- examples/
- requirements.txt

## Model
- Selected ablation: {finalist_info["selected_ablation"]}
- Encoder: {best_encoder}
- Temperature: {TEMPERATURE:.6f}
- Recommended mask threshold: {MASK_THRESHOLD:.2f}

## Outputs
- overlay
- binary mask
- explanation heatmap (HiResCAM / GradCAM++ / GradCAM)
- detections JSON
- short summary
"""

with open(DEMO_DIR / "app.py", "w") as f:
    f.write(app_py)
with open(DEMO_DIR / "requirements.txt", "w") as f:
    f.write(requirements_txt)
with open(DEMO_DIR / "README.md", "w") as f:
    f.write(readme_md)

print("Demo CAM package written to:", DEMO_DIR)

Demo CAM package written to: /kaggle/working/output/phase14a/phase14a_demo_cam_update


# Cell 15 — Figure manifest and final output check

In [15]:
manifest_rows = []

for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        manifest_rows.append({
            "relative_path": str(p.relative_to(OUT_DIR)),
            "suffix": p.suffix.lower(),
            "size_bytes": p.stat().st_size,
        })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(OUT_DIR / "phase14a_figure_manifest.csv", index=False)

print("Saved manifest:", OUT_DIR / "phase14a_figure_manifest.csv")
print("\nTop-level outputs:")
for p in sorted(OUT_DIR.glob("*")):
    print("-", p.name)

Saved manifest: /kaggle/working/output/phase14a/phase14a_figure_manifest.csv

Top-level outputs:
- phase14a_boundary_iou_barplot.png
- phase14a_boundary_topology_summary.csv
- phase14a_cam
- phase14a_cam_comparison_panel.png
- phase14a_cldice_barplot.png
- phase14a_demo_cam_update
- phase14a_failure_panel_examples.csv
- phase14a_failure_panel_supplement.png
- phase14a_figure_manifest.csv
- phase14a_gradcam_gallery.png
- phase14a_gradcampp_gallery.png
- phase14a_gt_cache
- phase14a_hirescam_gallery.png
- phase14a_image_level_confusion_matrix.png
- phase14a_image_level_report.json
- phase14a_main_panel_examples.csv
- phase14a_panels
- phase14a_per_image_metrics.csv
- phase14a_pixel_confusion_matrix_normalized.png
- phase14a_prob_cache
- phase14a_qualitative_panel_main_paper.png
- phase14a_source_metric_heatmap.png
- phase14a_source_metric_summary.csv
- phase14a_threshold_metric_heatmap.png
- phase14a_threshold_metrics.csv


In [16]:
import os
import shutil

source_dir = "/kaggle/working/output/phase14a"
zip_base = "/kaggle/working/phase14a_full_outputs"

if os.path.isdir(source_dir):
    zip_path = shutil.make_archive(zip_base, "zip", source_dir)
    print("ZIP created:", zip_path)
else:
    print("Folder not found:", source_dir)

ZIP created: /kaggle/working/phase14a_full_outputs.zip


In [17]:
import os
import shutil

source_dir = "/kaggle/working/output/phase14a/phase14a_demo_cam_update"
zip_base = "/kaggle/working/phase14a_demo_cam_update"

if os.path.isdir(source_dir):
    zip_path = shutil.make_archive(zip_base, "zip", source_dir)
    print("ZIP created:", zip_path)
else:
    print("Folder not found:", source_dir)

ZIP created: /kaggle/working/phase14a_demo_cam_update.zip
